In [1]:
import os
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm import tqdm
import chess
import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import math

load_dotenv()

if torch.cuda.is_available():
	print("PyTorch is using the GPU")
	GPUCount = torch.cuda.device_count()
	print(f"Found {GPUCount} GPUs")

	for i in range(GPUCount):
		print(f"GPU {i} found: {torch.cuda.get_device_name(i)}")

	device = torch.device("cuda:0")
else:
	print("PyTorch is using the CPU")
	device = torch.device("cpu")

print(f"Selected Device: {device}")

/home/nkminion/miniconda3/envs/PyTorchVenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch is using the GPU
Found 1 GPUs
GPU 0 found: NVIDIA GeForce RTX 5070 Laptop GPU
Selected Device: cuda:0


In [2]:
TOKEN = os.getenv("HFREAD")

if TOKEN is None:
	raise ValueError("Token not found")

Dataset = load_dataset(
	"Lichess/chess-position-evaluations",
	split='train',
	streaming=True,
	token=TOKEN
)

print('Dataset Loaded')

Dataset Loaded


In [3]:
def ProcessChessData(FENString,CPScore,MateScore):
	tensor = torch.zeros((14,8,8), dtype=torch.float32)
	board = chess.Board(FENString)

	PieceToLayer = {
		'P':0,'N':1,'B':2,'R':3,'Q':4,'K':5,
		'p':6,'n':7,'b':8,'r':9,'q':10,'k':11
	}

	for square in chess.SQUARES:
		piece = board.piece_at(square)

		if piece:
			symbol = piece.symbol()
			layer = PieceToLayer[symbol]

			row = 7-(square//8)
			col = square%8

			tensor[layer,row,col] = 1.0

	if board.turn == chess.WHITE:
		tensor[12,:,:] = 1.0

	if board.has_kingside_castling_rights(chess.WHITE):
		tensor[13,7,7] = 1.0
	if board.has_queenside_castling_rights(chess.WHITE):
		tensor[13,7,0] = 1.0
	if board.has_kingside_castling_rights(chess.BLACK):
		tensor[13,0,7] = 1.0
	if board.has_queenside_castling_rights(chess.BLACK):
		tensor[13,0,0] = 1.0

	if MateScore is not None:
		TargetScore = 1.0 if MateScore > 0 else -1.0
	else:
		CPScore = CPScore if CPScore is not None else 0
		TargetScore = math.tanh(CPScore/400.0)

	return tensor,TargetScore

In [4]:
class ResidualBlock(nn.Module):
	def __init__(self,NumChannels):
		super().__init__()
		self.conv1 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn1 = nn.BatchNorm2d(NumChannels)

		self.conv2 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn2 = nn.BatchNorm2d(NumChannels)

	def forward(self,x):
		residual = x
		
		x = f.relu(self.bn1(self.conv1(x)))

		x = self.bn2(self.conv2(x))

		x += residual

		return f.relu(x)
	
class ChessNet(nn.Module):
	def __init__(self):
		super().__init__()

		self.ConvInput = nn.Conv2d(in_channels=14,out_channels=128,kernel_size=3,padding=1)
		self.BnInput = nn.BatchNorm2d(128)

		self.ResTower = nn.Sequential(
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128),
			ResidualBlock(128)
		)

		self.ConvValue = nn.Conv2d(in_channels=128,out_channels=1,kernel_size=1)
		self.BnValue = nn.BatchNorm2d(1)

		self.flat = nn.Flatten()

		self.fc1 = nn.Linear(64,128)
		self.fc2 = nn.Linear(128,1)

	def forward(self,x):
		x = f.relu(self.BnInput(self.ConvInput(x)))

		x = self.ResTower(x)

		x = f.relu(self.BnValue(self.ConvValue(x)))
		x = self.flat(x)
		x = f.relu(self.fc1(x))

		x = torch.tanh(self.fc2(x))

		return x
	
model = ChessNet()
model.to(device)
print(model)

criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(),lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser,mode='min',factor=0.5,patience=3)

ChessNet(
  (ConvInput): Conv2d(14, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (BnInput): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (ResTower): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): ResidualBlo

In [5]:
class ChessIterableDataset(IterableDataset):
	def __init__(self,hfDataset):
		self.hfDataset = hfDataset

	def __iter__(self):
		for data in self.hfDataset:
			inputs,target = ProcessChessData(data['fen'],data['cp'],data['mate'])

			yield inputs,target

In [ ]:
print('Starting Stream...')

RunningLoss = 0.0
LogInterval = 100
BestLoss = float('inf')
MaxPatience = 10
PatienceCounter = 0

TrainDataset = ChessIterableDataset(Dataset)
TrainLoader = DataLoader(
	TrainDataset,
	batch_size=4096,
	num_workers=4,
	pin_memory=True,
	prefetch_factor=1,
	persistent_workers=True
)

model.train()
for BatchIdx,(inputs,targets) in enumerate(TrainLoader):
	inputs = inputs.to(device)
	targets = targets.to(device).float().unsqueeze(1)

	optimiser.zero_grad()

	predictions = model(inputs)

	loss = criterion(predictions,targets)
	loss.backward()
	optimiser.step()

	RunningLoss += loss.item()

	if (BatchIdx+1) % LogInterval == 0:
		AvgLoss = RunningLoss/LogInterval

		scheduler.step(AvgLoss)

		print(f"Batch {BatchIdx+1} | Average Loss: {AvgLoss:.6f}")

		if AvgLoss < BestLoss:
			print(f"Loss has decreased! Saving model!")
			BestLoss = AvgLoss

			torch.save(model.state_dict(),'ChessModel.pth')
			PatienceCounter = 0
		else:
			PatienceCounter += 1
			print(f'No improvement... Patience: {PatienceCounter}/{MaxPatience}')

			if (PatienceCounter >= MaxPatience):
				print('Patience limit reached. Stopping...')
				break

		RunningLoss = 0.0

Starting Stream...
Batch 100 | Average Loss: 0.251021
Loss has decreased! Saving model!
Batch 200 | Average Loss: 0.210551
Loss has decreased! Saving model!
Batch 300 | Average Loss: 0.187540
Loss has decreased! Saving model!
Batch 400 | Average Loss: 0.182259
Loss has decreased! Saving model!
Batch 500 | Average Loss: 0.176552
Loss has decreased! Saving model!
Batch 600 | Average Loss: 0.176043
Loss has decreased! Saving model!
Batch 700 | Average Loss: 0.164970
Loss has decreased! Saving model!
Batch 800 | Average Loss: 0.166025
No improvement... Patience: 1/10
Batch 900 | Average Loss: 0.163091
Loss has decreased! Saving model!
Batch 1000 | Average Loss: 0.157898
Loss has decreased! Saving model!
Batch 1100 | Average Loss: 0.163420
No improvement... Patience: 1/10
Batch 1200 | Average Loss: 0.155419
Loss has decreased! Saving model!
Batch 1300 | Average Loss: 0.144174
Loss has decreased! Saving model!
Batch 1400 | Average Loss: 0.147379
No improvement... Patience: 1/10
Batch 1500 | 

KeyboardInterrupt: 